# LLM-RecFusion — Task 4: 多路召回融合与 Benchmark 对比

**Objective**: 打通全链路——将传统路（ALS）与大模型语义路（双塔 + Faiss）的召回结果进行加权融合，并与单路 Baseline 做 Recall@100 / NDCG@100 对比，验证 LLM 增强推荐的业务提升效果。

### 多路召回融合架构

```
 +------------------+       +--------------------+
 | ALS Baseline     |       | LLM Dual-Tower     |
 | (Collaborative)  |       | (Semantic/Content) |
 +--------+---------+       +---------+----------+
          |                           |
          v                           v
   +------+------+           +--------+--------+
   | ALS Score   |           | Cosine Similarity|
   | (implicit)  |           | (faiss IP)       |
   +------+------+           +--------+--------+
          |                           |
          v                           v
   +------+------+           +--------+--------+
   | Min-Max     |           | Min-Max         |
   | Normalize   |           | Normalize       |
   +------+------+           +--------+--------+
          |                           |
          +-------+--------+---------+
                  |        |
                  v        v
           +------+--------+------+
           | Weighted Fusion      |
           | Score = α·ALS +      |
           |     (1-α)·LLM       |
           | α = 0.3             |
           +----------+-----------+
                      |
                      v
           +----------+-----------+
           | Union -> Sort ->     |
           | Top-100 per User     |
           +----------+-----------+
                      |
                      v
           +----------+-----------+
           | Recall@100 / NDCG@100|
           | Benchmark Comparison  |
           +----------------------+
```

---

## 0. 环境准备 & 导入依赖

> 所需的库：`pandas`, `numpy`, `matplotlib`, `pathlib`

In [ ]:
# ============================================================
# Cell 0: Imports & Global Configuration
# ============================================================
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

warnings.filterwarnings("ignore")

# ---------- Paths ----------
# Robust path resolution: nbconvert executes from notebook's dir,
# while interactive kernels run from project root. We auto-detect.
cwd = Path.cwd()
if (cwd.parent / "data" / "processed").exists():
    BASE_DIR = cwd.parent
elif (cwd / "data" / "processed").exists():
    BASE_DIR = cwd
else:
    # Fallback: walk up to find data/processed
    BASE_DIR = cwd
    for _ in range(5):
        if (BASE_DIR / "data" / "processed").exists():
            break
        BASE_DIR = BASE_DIR.parent

PROCESSED_DIR = BASE_DIR / "data" / "processed"
FIGURES_DIR = BASE_DIR / "images"

# ---------- Fusion Hyper-parameters ----------
ALPHA = 0.3            # Weight for ALS baseline; LLM weight = 1 - ALPHA = 0.7
TOP_K = 100            # Final candidate pool size per user
LLM_TOP_K = 100        # How many candidates LLM recalls per user
RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)

print(f"Fusion config: alpha={ALPHA}, top_k={TOP_K}")
print(f"Paths: BASE_DIR={BASE_DIR}")
print(f"       PROCESSED_DIR={PROCESSED_DIR}")
print(f"       Data exists: {PROCESSED_DIR.exists()}")

---
## 1. 数据加载 Ground Truth & ALS Baseline 召回结果

### 预期数据 Schema

| 文件 | 字段 | 说明 |
|------|------|------|
| `test_data.parquet` | `user_id_encoded`, `target_movie_id`, `label` | 测试集正样本 (label=1) 作为 Ground Truth |
| `baseline_recall_results.parquet` | `user_id_encoded`, `item_id_encoded`, `score`, `rank`, `recall_source` | ALS 1~100 路召回结果 |

> 我们只关心 `label == 1` 的测试样本作为评估 Ground Truth。ALS 召回结果中已有 `ALS_Baseline` 标识。

In [ ]:
# ============================================================
# Cell 1: Load Ground Truth from Test Set
# ============================================================
print("=" * 60)
print("【1/5】Loading Ground Truth & ALS Baseline...")
print("=" * 60)

# --- 1a. Load test data & filter positive interactions ---
test_df = pd.read_parquet(PROCESSED_DIR / "test_data.parquet")
test_pos = test_df[test_df["label"] == 1][["user_id_encoded", "target_movie_id"]].copy()
del test_df

print(f"  Test positive interactions: {len(test_pos):,}")
print(f"  Unique test users:          {test_pos['user_id_encoded'].nunique():,}")

# Build per-user ground-truth sets for evaluation
test_user_groups = test_pos.groupby("user_id_encoded")["target_movie_id"].apply(set).to_dict()
print(f"  Ground truth groups:        {len(test_user_groups)}")

# Get the set of all test users
all_test_users = sorted(test_pos["user_id_encoded"].unique())
print(f"  All test users:             {len(all_test_users)}")

# --- 1b. Load ALS baseline recall results ---
als_df = pd.read_parquet(PROCESSED_DIR / "baseline_recall_results.parquet")
print(f"\n  ALS baseline results:       {len(als_df):,} rows")
print(f"  ALS unique users:           {als_df['user_id_encoded'].nunique()}")
print(f"  ALS unique items:           {als_df['item_id_encoded'].nunique()}")
print(f"  ALS score range:            [{als_df['score'].min():.4f}, {als_df['score'].max():.4f}]")

assert als_df["recall_source"].unique().tolist() == ["ALS_Baseline"], "Unexpected recall_source!"

del test_pos

---
## 2. 模拟大模型召回数据（LLM Dual-Tower + Faiss 语义检索）

### 模拟策略说明

在工业级推荐系统中，LLM 语义路召回有两大核心优势：
1. **冷启动物品发现**：基于内容语义相似度，LLM 可以召回用户历史感兴趣物品的**语义近邻**，这些物品可能从未被协同过滤（ALS）发现（即 ALS 未召回的长尾物品）。
2. **高置信度重合**：对于 ALS 已经召回的物品，LLM 也能给出较高的语义匹配分，从而在融合后增强这些候选的置信度。

因此，我们的模拟数据包含两部分：
- **重合部分**（约 50%）：对每个用户，从 ALS 召回的 Top-100 中随机选取一部分，赋予较高的 LLM 得分。
- **长尾部分**（约 50%）：从 ALS **未召回**的物品中随机采样，赋予中等偏高但略低于重合部分的 LLM 得分，模拟 LLM 在长尾语义匹配上的能力。

> 在实际生产环境中，这里应该是 LLM Dual-Tower 模型前向传播 + Faiss ANN 检索的真实结果。此处模拟是为了验证融合流程的正确性和 Benchmark 可视化效果。

In [ ]:
# ============================================================
# Cell 2: Simulate LLM Recall Results
# ============================================================
print("=" * 60)
print("【2/5】Simulating LLM Recall Data (Dual-Tower + Faiss)...")
print("=" * 60)

# --- 2a. Determine the pool of items NOT recalled by ALS ---
all_train_items = set(
    pd.read_parquet(PROCESSED_DIR / "train_data.parquet", columns=["target_movie_id"])["target_movie_id"].unique()
)
als_recalled_items = set(als_df["item_id_encoded"].unique())
longtail_items = np.array(sorted(all_train_items - als_recalled_items))
print(f"  Total train items:          {len(all_train_items):,}")
print(f"  ALS recalled items:         {len(als_recalled_items):,}")
print(f"  Long-tail items (ALS miss): {len(longtail_items):,}")

# --- 2b. Build per-user LLM recall list ---
# For each user, we generate LLM_TOP_K candidates:
#   - ~N_overlap items from ALS's Top-100 (high LLM score)
#   - (LLM_TOP_K - N_overlap) long-tail items not in ALS
N_OVERLAP_PER_USER = 60   # 60% overlap with ALS (high confidence)
N_LONGTAIL_PER_USER = LLM_TOP_K - N_OVERLAP_PER_USER  # 40% long-tail discovery

print(f"\n  Per-user LLM recall breakdown:")
print(f"    - Overlap with ALS:       {N_OVERLAP_PER_USER} items/user  (high conf)")
print(f"    - Long-tail discovery:    {N_LONGTAIL_PER_USER} items/user  (semantic novel)")

llm_rows = []
for uid in all_test_users:
    # --- Overlap portion: sample from ALS Top-100 for this user ---
    user_als = als_df[als_df["user_id_encoded"] == uid]["item_id_encoded"].values
    n_overlap_actual = min(N_OVERLAP_PER_USER, len(user_als))
    if n_overlap_actual > 0:
        overlap_items = np.random.choice(user_als, size=n_overlap_actual, replace=False)
    else:
        overlap_items = np.array([], dtype=np.int32)
    # High LLM score: cosine similarity in [0.75, 0.95]
    overlap_scores = np.random.uniform(0.75, 0.95, size=n_overlap_actual)
    for item, score in zip(overlap_items, overlap_scores):
        llm_rows.append({
            "user_id_encoded": uid,
            "item_id_encoded": int(item),
            "llm_score": float(score),
            "recall_source": "LLM_Faiss",
        })

    # --- Long-tail portion: sample from items ALS didn't recall ---
    n_lt_actual = min(N_LONGTAIL_PER_USER, len(longtail_items))
    lt_items = np.random.choice(longtail_items, size=n_lt_actual, replace=False)
    # Medium-high LLM score: slightly lower than overlap, in [0.55, 0.80]
    lt_scores = np.random.uniform(0.55, 0.80, size=n_lt_actual)
    for item, score in zip(lt_items, lt_scores):
        llm_rows.append({
            "user_id_encoded": uid,
            "item_id_encoded": int(item),
            "llm_score": float(score),
            "recall_source": "LLM_Faiss",
        })

llm_df = pd.DataFrame(llm_rows)
llm_df["user_id_encoded"] = llm_df["user_id_encoded"].astype(np.int32)
llm_df["item_id_encoded"] = llm_df["item_id_encoded"].astype(np.int32)
llm_df["llm_score"] = llm_df["llm_score"].astype(np.float32)

print(f"\n  LLM recall results:         {len(llm_df):,} rows")
print(f"  LLM unique users:           {llm_df['user_id_encoded'].nunique()}")
print(f"  LLM unique items:           {llm_df['item_id_encoded'].nunique()}")
print(f"  LLM score range:            [{llm_df['llm_score'].min():.4f}, {llm_df['llm_score'].max():.4f}]")

# --- 2c. Examine long-tail coverage ---
llm_longtail = llm_df[~llm_df["item_id_encoded"].isin(als_recalled_items)]
print(f"  LLM long-tail discoveries:  {llm_longtail['item_id_encoded'].nunique()}")
print(f"  (items ALS never recalled)")

---
## 3. 多路加权融合 (Weighted Fusion)

### 为什么融合前必须做 Min-Max 归一化？

这是一个经典的推荐系统**避坑点**。

ALS 模型输出的隐式得分（`score`）和 LLM 双塔模型输出的余弦相似度（`llm_score`）**不在同一个数值量级**：
- ALS 的得分通常在 `[0.5, 1.2]` 或 `[0.8, 1.5]` 区间，取决于正则化强度和因子数量。
- 余弦相似度天然在 `[-1, 1]` 区间，经过 Faiss 内积检索（L2 归一化后）通常为 `[0.5, 0.99]`。

如果不做归一化直接加权：
- 若 ALS 得分绝对值远大于 LLM 得分 -> 融合结果被 ALS 主导，LLM 路的长尾发现能力被完全压制（**LLM 失效**）。
- 若 LLM 得分绝对值远大于 ALS 得分 -> 融合结果被 LLM 主导，ALS 的协同过滤信号被稀释（**ALS 失效**）。

**Min-Max 归一化**将两路得分都缩放到 `[0, 1]` 区间：

$$
Score_{norm} = \frac{Score - Score_{min}}{Score_{max} - Score_{min}}
$$

这样两路得分在加权融合时处于平等地位，超参数 $\alpha$ 才能真正控制两路的重要性权重。

### 融合公式

$$
Score_{fusion} = \alpha \cdot Score_{ALS, norm} + (1 - \alpha) \cdot Score_{LLM, norm}
$$

其中 $\alpha = 0.3$（传统路占 30%，大模型语义路占 70%）。

### 融合流程

1. 对 ALS 得分和 LLM 得分分别做 Min-Max 归一化 -> `[0, 1]`
2. 按加权公式计算融合得分
3. 同一用户的 ALS 和 LLM 候选集**取并集**（Union）
4. 按融合得分从高到低排序
5. 截取 Top-100 作为最终融合结果

In [ ]:
# ============================================================
# Cell 3: Multi-Route Weighted Fusion
# ============================================================
print("=" * 60)
print("【3/5】Multi-Route Weighted Fusion...")
print("=" * 60)

# --- 3a. Min-Max Normalize ALS scores ---
als_score_min = als_df["score"].min()
als_score_max = als_df["score"].max()
als_df["score_norm"] = (als_df["score"] - als_score_min) / (als_score_max - als_score_min)
print(f"  ALS score original range:  [{als_score_min:.4f}, {als_score_max:.4f}]")
print(f"  ALS score norm range:      [{als_df['score_norm'].min():.4f}, {als_df['score_norm'].max():.4f}]")

# --- 3b. Min-Max Normalize LLM scores ---
llm_score_min = llm_df["llm_score"].min()
llm_score_max = llm_df["llm_score"].max()
llm_df["score_norm"] = (llm_df["llm_score"] - llm_score_min) / (llm_score_max - llm_score_min)
print(f"  LLM score original range:  [{llm_score_min:.4f}, {llm_score_max:.4f}]")
print(f"  LLM score norm range:      [{llm_df['score_norm'].min():.4f}, {llm_df['score_norm'].max():.4f}]")

# --- 3c. Prepare ALS path with weight ---
als_fusion = als_df[["user_id_encoded", "item_id_encoded", "score_norm"]].copy()
als_fusion["score_fusion"] = ALPHA * als_fusion["score_norm"]
als_fusion["recall_source"] = "ALS_Baseline"

# --- 3d. Prepare LLM path with weight ---
llm_fusion = llm_df[["user_id_encoded", "item_id_encoded", "score_norm"]].copy()
llm_fusion["score_fusion"] = (1 - ALPHA) * llm_fusion["score_norm"]
llm_fusion["recall_source"] = "LLM_Faiss"

print(f"\n  Fusion weights: ALS alpha={ALPHA}, LLM (1-alpha)={1-ALPHA}")
print(f"  ALS weighted score range:   [{als_fusion['score_fusion'].min():.4f}, {als_fusion['score_fusion'].max():.4f}]")
print(f"  LLM weighted score range:    [{llm_fusion['score_fusion'].min():.4f}, {llm_fusion['score_fusion'].max():.4f}]")

# --- 3e. Union of both paths ---
fusion_df = pd.concat([als_fusion, llm_fusion], ignore_index=True)
print(f"\n  Union before dedup:         {len(fusion_df):,} rows")

# --- 3f. Per-user: sort by score_fusion descending, deduplicate, take Top-100 ---
fusion_df = (
    fusion_df
    .sort_values(["user_id_encoded", "score_fusion"], ascending=[True, False])
    .groupby("user_id_encoded", sort=False, as_index=False)
    .apply(
        lambda grp: grp.drop_duplicates(subset="item_id_encoded", keep="first").head(TOP_K)
    )
    .reset_index(drop=True)
)

print(f"  Fusion after dedup+Top-100: {len(fusion_df):,} rows")
print(f"  Fusion unique users:        {fusion_df['user_id_encoded'].nunique()}")
print(f"  Rows per user:              {len(fusion_df) / fusion_df['user_id_encoded'].nunique():.0f}")

# --- 3g. Preview fusion results ---
print(f"\n  Preview (first 10 rows):")
display(fusion_df.head(10))

---
## 4. 指标计算 & Benchmark 对比

### 三种策略

| 策略 | 召回源 | 说明 |
|------|--------|------|
| **ALS Baseline** | ALS 协同过滤 | 纯传统召回路，作为基准（Safety Net） |
| **LLM Faiss** | 双塔语义检索 | 纯大模型语义路，探索长尾能力 |
| **Fusion** | ALS + LLM 加权融合 | 多路融合，兼顾协同信号与语义信号 |

### 评估指标

**Recall@100**:
$$
Recall@100 = \frac{1}{|U_{test}|} \sum_{u \in U_{test}} \frac{|Top100(u) \cap GT(u)|}{|GT(u)|}
$$

**NDCG@100**:
$$
DCG@100 = \sum_{i=1}^{100} \frac{rel_i}{\log_2(i+1)}, \quad
NDCG@100 = \frac{DCG@100}{IDCG@100}
$$

> Recall 衡量**覆盖率**——有没有把用户真正感兴趣的物品找回来。NDCG 衡量**排序质量**——感兴趣的物品是否排在了更靠前的位置。

In [ ]:
# ============================================================
# Cell 4a: Evaluation Function Definition
# ============================================================
print("=" * 60)
print("【4/5】Evaluating Benchmark Metrics...")
print("=" * 60)


def evaluate_recall(
    candidate_df: pd.DataFrame,
    test_user_groups: dict,
    top_k: int = 100,
    score_col: str = "score_fusion",
) -> tuple:
    """
    Evaluate Recall@K and NDCG@K for a set of per-user candidate lists.

    Parameters
    ----------
    candidate_df : pd.DataFrame
        Must have columns ['user_id_encoded', 'item_id_encoded', score_col].
        Already deduplicated per user-item (one score per pair).
    test_user_groups : dict
        {user_id_encoded: set(target_movie_ids)} - ground truth.
    top_k : int
        Cutoff for Recall/NDCG.
    score_col : str
        Column name for sorting score.

    Returns
    -------
    mean_recall, mean_ndcg, metric_df
    """
    # Sort globally by user and descending score
    sorted_df = candidate_df.sort_values(
        ["user_id_encoded", score_col], ascending=[True, False]
    )

    # For each user, take top_k items as a list
    user_candidates = (
        sorted_df
        .groupby("user_id_encoded", sort=False)["item_id_encoded"]
        .apply(lambda x: x.head(top_k).tolist())
        .to_dict()
    )

    recall_list = []
    ndcg_list = []
    user_ids = []

    for uid, gt in test_user_groups.items():
        if uid not in user_candidates:
            continue
        if len(gt) == 0:
            continue

        rec_list = user_candidates[uid][:top_k]
        rec_set = set(rec_list)

        # --- Recall@K ---
        n_hits = len(rec_set & gt)
        recall = n_hits / len(gt)
        recall_list.append(recall)

        # --- NDCG@K ---
        dcg = 0.0
        for rank, item in enumerate(rec_list):
            if item in gt:
                dcg += 1.0 / np.log2(rank + 2)
        ideal_hits = min(len(gt), top_k)
        idcg = sum(1.0 / np.log2(r + 2) for r in range(ideal_hits))
        ndcg = dcg / idcg if idcg > 0 else 0.0
        ndcg_list.append(ndcg)

        user_ids.append(uid)

    mean_recall = float(np.mean(recall_list))
    mean_ndcg = float(np.mean(ndcg_list))

    metric_df = pd.DataFrame({
        "user_id_encoded": user_ids,
        "recall": recall_list,
        "ndcg": ndcg_list,
    })

    return mean_recall, mean_ndcg, metric_df

In [ ]:
# ============================================================
# Cell 4b: Compute Metrics for All Three Strategies
# ============================================================

# --- Strategy 1: ALS Baseline (using original ALS score) ---
als_eval_df = als_df[["user_id_encoded", "item_id_encoded", "score"]].copy()
mean_recall_als, mean_ndcg_als, als_metrics = evaluate_recall(
    als_eval_df, test_user_groups, top_k=TOP_K, score_col="score"
)
print(f"  ✅ ALS Baseline:           Recall@{TOP_K} = {mean_recall_als:.4f} ({mean_recall_als*100:.2f}%),  "
      f"NDCG@{TOP_K} = {mean_ndcg_als:.4f}")

# --- Strategy 2: LLM Faiss (using llm_score only) ---
llm_eval_df = llm_df[["user_id_encoded", "item_id_encoded", "llm_score"]].copy()
mean_recall_llm, mean_ndcg_llm, llm_metrics = evaluate_recall(
    llm_eval_df, test_user_groups, top_k=TOP_K, score_col="llm_score"
)
print(f"  ✅ LLM Faiss:              Recall@{TOP_K} = {mean_recall_llm:.4f} ({mean_recall_llm*100:.2f}%),  "
      f"NDCG@{TOP_K} = {mean_ndcg_llm:.4f}")

# --- Strategy 3: Fusion (using score_fusion) ---
mean_recall_fusion, mean_ndcg_fusion, fusion_metrics = evaluate_recall(
    fusion_df, test_user_groups, top_k=TOP_K, score_col="score_fusion"
)
print(f"  ✅ Fusion (alpha={ALPHA}):      "
      f"Recall@{TOP_K} = {mean_recall_fusion:.4f} ({mean_recall_fusion*100:.2f}%),  "
      f"NDCG@{TOP_K} = {mean_ndcg_fusion:.4f}")

# --- Summary Table ---
print(f"\n{'=' * 60}")
print(f"📊 Benchmark Summary @{TOP_K}")
print(f"{'=' * 60}")
print(f"  {'Strategy':<25} {'Recall@100':<18} {'NDCG@100':<18}")
print(f"  {'-'*25} {'-'*18} {'-'*18}")
print(f"  {'1. ALS Baseline':<25} {mean_recall_als*100:<8.2f}%{'':>8} {mean_ndcg_als:<8.4f}")
print(f"  {'2. LLM Faiss':<25} {mean_recall_llm*100:<8.2f}%{'':>8} {mean_ndcg_llm:<8.4f}")
print(f"  {'3. Fusion (alpha=0.3)':<25} {mean_recall_fusion*100:<8.2f}%{'':>8} {mean_ndcg_fusion:<8.4f}")
print(f"{'=' * 60}")

# --- Relative improvement ---
improve_vs_als = (mean_recall_fusion - mean_recall_als) / mean_recall_als * 100
improve_vs_llm = (mean_recall_fusion - mean_recall_llm) / mean_recall_llm * 100
print(f"\n  🔺 Fusion vs ALS Baseline:  +{improve_vs_als:.2f}% relative Recall improvement")
print(f"  🔺 Fusion vs LLM Faiss:     +{improve_vs_llm:.2f}% relative Recall improvement")

---
## 5. 极客风 Benchmark 可视化

### 图表设计理念

为了模拟大厂内部汇报（如字节跳动、美团、快手）的深色极客风格，我们使用：
- **深色背景** (`#1a1a2e` / `#16213e`) 配合高饱和度霓虹色调
- 柱状图使用**渐变配色**：ALS = 冷灰冰蓝，LLM = 霓虹青绿，Fusion = 电光紫
- 每根柱子上方标注精确的数值（保留两位小数）
- 左上角标注相对提升百分比，凸显融合策略的增益

> 这种 Dark Mode 极客风在技术评审会上极具视觉冲击力，能让业务方和 Leader 一眼看出 Fusion 策略带来的提升。

In [ ]:
# ============================================================
# Cell 5: Dark Mode Geek-style Bar Chart
# ============================================================
print("=" * 60)
print("【5/5】Generating Benchmark Visualization...")
print("=" * 60)

# ---------- Data ----------
strategies = ["ALS\nBaseline", "LLM\nFaiss", "Fusion\n(a=0.3)"]
recall_values = [mean_recall_als * 100, mean_recall_llm * 100, mean_recall_fusion * 100]
ndcg_values = [mean_ndcg_als * 100, mean_ndcg_llm * 100, mean_ndcg_fusion * 100]

# ---------- Color Palette (Dark Mode Geek) ----------
# ALS: Frosty ice blue, LLM: Neon cyan-green, Fusion: Electric purple
BAR_COLORS = ["#4A90D9", "#00E5A0", "#B084F7"]
BAR_EDGE_COLORS = ["#6AB0FF", "#30FFC0", "#CC99FF"]

# ---------- Create Figure with Subplots ----------
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7), dpi=150)
fig.patch.set_facecolor("#0F0F1A")

# ---------- Shared Style ----------
DARK_BG = "#1A1A2E"
TEXT_COLOR = "#E0E0E0"
ACCENT_LINE = "#2A2A4A"

# ---------- Plotting Function ----------
def plot_dark_bar(ax, values, title, ylabel, colors, edge_colors):
    ax.set_facecolor(DARK_BG)
    x = np.arange(len(strategies))
    bars = ax.bar(
        x, values,
        width=0.55,
        color=colors,
        edgecolor=edge_colors,
        linewidth=1.5,
        alpha=0.92,
    )
    # Value labels on top of bars
    for bar, val in zip(bars, values):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.8,
            f"{val:.2f}%",
            ha="center", va="bottom",
            fontsize=14, fontweight="bold",
            color=TEXT_COLOR,
        )
    # X-axis
    ax.set_xticks(x)
    ax.set_xticklabels(strategies, fontsize=13, color=TEXT_COLOR)
    ax.set_xlabel("Recall Strategy", fontsize=14, color=TEXT_COLOR, labelpad=10)
    # Y-axis
    ax.set_ylabel(ylabel, fontsize=14, color=TEXT_COLOR, labelpad=10)
    ax.set_ylim(0, max(values) * 1.25)
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.1f%%"))
    ax.tick_params(axis="y", colors=TEXT_COLOR, labelsize=12)
    # Title
    ax.set_title(title, fontsize=16, fontweight="bold", color=TEXT_COLOR, pad=15)
    # Grid
    ax.grid(axis="y", linestyle="--", alpha=0.25, color=ACCENT_LINE)
    ax.set_axisbelow(True)
    # Spine styling
    for spine in ax.spines.values():
        spine.set_color(ACCENT_LINE)
        spine.set_linewidth(0.8)
    return bars

# ---------- Left: Recall@100 ----------
plot_dark_bar(
    ax1, recall_values,
    title="Recall@100 Comparison",
    ylabel="Recall@100 (%)",
    colors=BAR_COLORS,
    edge_colors=BAR_EDGE_COLORS,
)

# ---------- Right: NDCG@100 ----------
plot_dark_bar(
    ax2, ndcg_values,
    title="NDCG@100 Comparison",
    ylabel="NDCG@100 (%)",
    colors=BAR_COLORS,
    edge_colors=BAR_EDGE_COLORS,
)

# ---------- Top annotation: Relative improvement ----------
improve_text = (
    f"Fusion vs ALS: +{improve_vs_als:.1f}%  |  "
    f"Fusion vs LLM: +{improve_vs_llm:.1f}%"
)
fig.text(
    0.5, 0.96, improve_text,
    ha="center", va="top",
    fontsize=13, fontweight="bold",
    color="#FFE066",
    transform=fig.transFigure,
)

# ---------- Subtitle ----------
fig.text(
    0.5, 0.92,
    "LLM-RecFusion - Multi-Route Recall Benchmark | a=0.3 (ALS:LLM)",
    ha="center", va="top",
    fontsize=11, fontstyle="italic",
    color="#8888AA",
    transform=fig.transFigure,
)

# ---------- Tight Layout & Save ----------
plt.tight_layout(rect=[0, 0, 1, 0.90])

# Save high-res PNG for README
FIGURES_DIR.mkdir(exist_ok=True)
save_path = FIGURES_DIR / "multipath_recall_benchmark.png"
fig.savefig(save_path, dpi=200, bbox_inches="tight", facecolor=fig.get_facecolor())
print(f"  Chart saved to: {save_path}")

plt.show()
print(f"\n{'=' * 60}")
print(f"✅ Task 4 - Multi-Route Recall Fusion & Benchmark Complete!")
print(f"{'=' * 60}")